# V5 Feature Engineering

Runs the V5 feature engineer on Google Colab. Outputs per-season feature Parquets to Drive.

**Expected runtime:** 1-2 hours on Colab Pro high-RAM CPU.

## Before running

Ensure this structure exists in Google Drive:
```
My Drive/SportsAnalyzer/
├── data/nfl/          ← 13 dataset folders (Parquet files)
├── src/nfl/features/v5/   ← upload the v5 package (config.py, master_table.py, etc.)
└── output/features/   ← will be created by this notebook
```

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set paths and verify they exist
import sys
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/SportsAnalyzer'
DATA_DIR = f'{DRIVE_ROOT}/data/nfl'
OUTPUT_DIR = f'{DRIVE_ROOT}/output/features'
CODE_ROOT = DRIVE_ROOT

# Add code root to Python path so `from src.nfl.features.v5 import ...` works
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

# Verify paths
assert Path(DATA_DIR).exists(), f'Data directory not found: {DATA_DIR}'
assert Path(f'{CODE_ROOT}/src/nfl/features/v5/engineer.py').exists(), \
    f'Code not uploaded. Copy src/nfl/features/v5/ to {CODE_ROOT}/src/nfl/features/v5/'

# Ensure __init__.py files exist for package imports on Colab
for init_path in [
    f'{CODE_ROOT}/src/__init__.py',
    f'{CODE_ROOT}/src/nfl/__init__.py',
    f'{CODE_ROOT}/src/nfl/features/__init__.py',
]:
    Path(init_path).touch(exist_ok=True)

print(f'Data: {DATA_DIR}')
print(f'Code: {CODE_ROOT}/src/nfl/features/v5/')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# 3. Check available RAM (should be high-RAM ~50GB on Colab Pro)
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'Available RAM: {ram_gb:.1f} GB')
if ram_gb < 20:
    print('WARNING: Not using high-RAM runtime. Feature engineering may fail or be slow.')
    print('Change via Runtime > Change runtime type > High-RAM')

In [ ]:
# 4. Run feature engineering for all seasons
from src.nfl.features.v5 import build_features

SEASONS = list(range(2018, 2026))  # 2018-2025 (2018-2019 are warm-up)

df = build_features(
    data_dir=DATA_DIR,
    seasons=SEASONS,
    output_dir=OUTPUT_DIR,
    verbose=True,
)

print(f'\nDone: {len(df):,} rows, {len(df.columns)} columns')

In [ ]:
# 5. Verify outputs
import os
output_v5 = f'{OUTPUT_DIR}/v5'
total_size = 0
print(f'Feature files in {output_v5}:')
for f in sorted(os.listdir(output_v5)):
    size_mb = os.path.getsize(f'{output_v5}/{f}') / 1e6
    total_size += size_mb
    print(f'  {f}: {size_mb:.1f} MB')
print(f'\nTotal: {total_size:.1f} MB across {len(os.listdir(output_v5))} files')

In [ ]:
# 6. Spot-check: Mahomes 2024 Week 5 features (should have non-null rolling avg)
import pandas as pd
df_2024 = pd.read_parquet(f'{output_v5}/features_2024.parquet')
mahomes_w5 = df_2024[
    (df_2024['player_id'] == '00-0033873') & (df_2024['week'] == 5)
]
if len(mahomes_w5) > 0:
    row = mahomes_w5.iloc[0]
    print(f'Mahomes 2024 W5:')
    print(f'  rolling_avg_passing_yards: {row["rolling_avg_passing_yards"]:.1f}')
    print(f'  rolling_avg_fantasy_points_ppr: {row["rolling_avg_fantasy_points_ppr"]:.1f}')
    print(f'  game_script_index: {row["game_script_index"]}')
    print(f'  injury_severity: {row["injury_severity"]}')
else:
    print('Mahomes W5 not found — check that 2024 data loaded correctly')

## Done

If cell 6 shows reasonable values (passing yards ~200-300, fantasy points ~15-25), features generated successfully.

Report back in chat: "features done" with the output of cell 5 (file sizes).